### Операции над тензорами

Выполните небольшие упражнения, которые помогут закрепить ваши навыки манипуляций над тензорами. Вам нужно дописать код в местах с `...` и добиться того, чтобы все тесты проходили.

Если захочется дополнительной тренировки, в репозитории https://github.com/rougier/numpy-100 есть много упражнений с массивами NumPy, этот опыт отлично транслируется на работу с тензорами в Pytorch и Jax.

In [31]:
import torch

#### 1.1 Среднее значение по столбцам

In [32]:
torch.manual_seed(42)
x = torch.randint(10, size=(2, 3)).float()
x

tensor([[2., 7., 6.],
        [4., 6., 5.]])

In [33]:
mean_by_column = x.mean(axis=0)
assert torch.allclose(
    mean_by_column, _expected := torch.tensor([3.0, 6.5, 5.5])
), f"{mean_by_column} != {_expected}"

#### 1.2. Взвешенное среднее
В тензоре `w` находятся веса для расчёта взвешенных средних значений тензора `x` по строкам:

$$\bar{x}_i = \sum_{j=1}^n x_{ij} w_{ij}$$

Найдите эти взвешенные средние.

**NB:** Обратите внимание, что веса $w_i$, дающие в сумме по столбцам единицу, мы получили применением функции `torch.softmax` (или метода `.softmax`) к исходным ненормализованным весам. Эта особая функция нам позже пригодится в задаче многоклассовой классификации, чтобы моделировать вероятностное распределение над возможными классами.

In [34]:
torch.manual_seed(42)
x = torch.randint(10, size=(2, 3)).float()
w = torch.randint(10, size=(2, 3)).float().softmax(dim=1)
print(x)
print(w)

tensor([[2., 7., 6.],
        [4., 6., 5.]])
tensor([[0.0177, 0.9647, 0.0177],
        [0.0066, 0.9756, 0.0179]])


In [35]:
w_avg = torch.tensor(((x[0] @ w[0]), (x[1] @ w[1])))
assert torch.allclose(
    w_avg, _expected := torch.tensor([6.8940, 5.9690])
), f"{w_avg} != {_expected}"

#### 1.3. Зануление всех нечётных элементов в наборе векторов

В тензоре `x` — 4 вектора размера 7. В каждом векторе сделайте чётные элементы нулевыми.

Способов много, попробуйте разные!

In [36]:
x = torch.arange(28).view(4, 7) + 1
print(x)

tensor([[ 1,  2,  3,  4,  5,  6,  7],
        [ 8,  9, 10, 11, 12, 13, 14],
        [15, 16, 17, 18, 19, 20, 21],
        [22, 23, 24, 25, 26, 27, 28]])


In [37]:
# способ 1: использование broadcasting и маски, которая закрывает чётные элементы
mask = torch.tensor([(i+1) % 2 for i in range(7)])
print(mask)
x = x * mask

assert torch.allclose(
    x, _expected := torch.tensor([
        [ 1,  0,  3,  0,  5,  0,  7],
        [ 8,  0, 10,  0, 12,  0, 14],
        [15,  0, 17,  0, 19,  0, 21],
        [22,  0, 24,  0, 26,  0, 28]
    ])), f"{x} != {_expected}"

tensor([1, 0, 1, 0, 1, 0, 1])


In [38]:
# способ 2: изменение на месте с помощью среза
x = torch.arange(28).view(4, 7) + 1
x[:, [1, 3, 5]] = 0  # вставляем нули в нужные места

assert torch.allclose(
    x, _expected := torch.tensor([
        [ 1,  0,  3,  0,  5,  0,  7],
        [ 8,  0, 10,  0, 12,  0, 14],
        [15,  0, 17,  0, 19,  0, 21],
        [22,  0, 24,  0, 26,  0, 28]
    ])), f"{x} != {_expected}"

#### 1.4. Матрица попарных расстояний

Даны две матрицы `x` и `y`, нужно получить матрицу `d`, где `d[i, j]` - евклидово расстояние между векторами `x[i]` (строка `i` матрицы `x`) и `y[j]` (строка `j` матрицы `y`).

Подсказка 1: воспользуйтесь broadcasting и добавлением размерностей в исходные тензоры. Для этого могут пригодиться методы `.view` и `.unsqueeze`

Подсказка 2: можно не считать евклидово расстояние вручную, есть функция `torch.linalg.norm` (или метод `.norm`), в которую можно подать векторы поэлементных разностей (их вы и получите с помощью broadcasting, если правильно измените размерности)

In [39]:
torch.manual_seed(42)
x = torch.randint(10, size=(2, 3)).float()
y = torch.randint(10, size=(3, 3)).float()
print(x)
print(y)

tensor([[2., 7., 6.],
        [4., 6., 5.]])
tensor([[0., 4., 0.],
        [3., 8., 4.],
        [0., 4., 1.]])


In [40]:
pdist = (x.unsqueeze(1) - y.unsqueeze(0)).norm(p=2, dim=-1)
assert torch.allclose(
    pdist,
    _expected := torch.tensor(
        [
            [7.0000, 2.4495, 6.1644],
            [6.7082, 2.4495, 6.0000]
        ]
    ),
), f"{pdist} != {_expected}"

#### 1.5. Умножение матриц на векторы

В тензоре `m` - две матрицы, нужно сделать тензор, в котором i-й элемент - результат умножения матрицы `m[i]` на вектор `x[i]`.

Это можно было бы сделать так: `torch.stack([m[i] @ x[i] for i in len(m)])`.

Найдите решение без цикла.

In [41]:
torch.manual_seed(42)
x = torch.randint(10, size=(2, 3)).float()
m = torch.randint(10, size=(2, 3, 3)).float()
print(m)
print(x)

tensor([[[0., 4., 0.],
         [3., 8., 4.],
         [0., 4., 1.]],

        [[2., 5., 5.],
         [7., 6., 9.],
         [6., 3., 1.]]])
tensor([[2., 7., 6.],
        [4., 6., 5.]])


In [42]:
matmul = (m @ x.unsqueeze(-1)).view(2, 3)
assert torch.allclose(
    matmul, _expected := torch.tensor([[28.0, 86.0, 34.0], [63.0, 109.0, 47.0]])
), f"{matmul} != {_expected}"